# TERA — Travel Expense Review Assistant

## Prototype Development

This notebook is used for the initial development and evaluation of the AI components of TERA.

TERA is a research prototype developed as part of a Bachelor's thesis at the University of Bayreuth. The goal of the project is to investigate how artificial intelligence can support the review of international travel expense reimbursements.

The initial experiments focus on:
- processing travel-related documents,
- extracting structured information,
- identifying inconsistencies or potentially relevant issues,
- evaluating different AI-based approaches.

The final reimbursement decision remains with the human reviewer.

## Use Case 1 — Extract Structured Information from a Machine-Readable Hotel Invoice

**Goal:**
Extract relevant reimbursement-related information from a hotel invoice that contains machine-readable text.

**Input:**
A machine-readable PDF hotel invoice with an embedded text layer. No OCR is required.

**Method:**
Extract the text directly from the PDF and provide the extracted content to an AI model using a structured extraction prompt.

**Expected output:**
A structured representation containing relevant fields such as hotel name, invoice date, stay dates, currency, and total amount.

**Evaluation:**
Compare the AI-extracted values with manually identified ground-truth values from the source document.

In [158]:
import pymupdf
import json
import requests
import pandas as pd
import time
from pydantic import BaseModel
from pathlib import Path

### 1. Load and Inspect the Document

In [159]:
DATA_DIR = Path("data")
document_path = DATA_DIR / "hotel_invoice.pdf"

doc = pymupdf.open(document_path)

text = "\n".join(page.get_text() for page in doc)

print(text[:3000])

SYNTHETIC TEST INVOICE | Machine-readable PDF with selectable text
Page 1
NORTHSTAR HOTEL BERLIN
Fictional property - synthetic test document
Alexanderplatz 99
10178 Berlin, Germany
+49 30 5550 0199
billing@northstar-example.test
INVOICE
PAID
INVOICE NUMBER
NSB-2026-0914-1042
INVOICE DATE
14 September 2026
GUEST NAME
Alex Morgan
BOOKING REFERENCE
TERA-DEMO-74291
CHECK-IN
10 September 2026, 15:00
CHECK-OUT
14 September 2026, 11:00
STAY
4 room nights
ROOM
Deluxe King - 508
CHARGES
Description
Qty
Unit rate
Net
Tax
Amount
Accommodation - Deluxe King
4 nights
EUR 140.00
EUR 560.00
VAT 7%
EUR 599.20
Breakfast
4
EUR 15.00
EUR 60.00
VAT 19%
EUR 71.40
Berlin city tax (7.5% of room net)
1
EUR 42.00
EUR 42.00
Exempt
EUR 42.00
Tax summary
Taxable base
Tax amount
VAT 7% - accommodation
EUR 560.00
EUR 39.20
VAT 19% - breakfast
EUR 60.00
EUR 11.40
City tax - exempt from VAT
EUR 42.00
EUR 0.00
Subtotal (net + city tax)
EUR 662.00
VAT total
EUR 50.60
TOTAL AMOUNT
EUR 712.60
Currency
EUR
PAYMENT METHOD

### 2. Define the Target Extraction Schema

Not final schema. This is a preliminary version and will be updated based on the results of the experiments.

In [160]:
class HotelInvoiceExtraction(BaseModel):
    hotel_name: str | None
    hotel_address: str | None
    invoice_number: str | None
    invoice_date: str | None
    guest_name: str | None
    check_in_date: str | None
    check_out_date: str | None
    number_of_nights: int | None
    currency: str | None
    room_rate: float | None
    city_tax: float | None
    total_amount: float | None
    payment_method: str | None

### 3. Define Ground Truth

In [161]:
ground_truth = {
    "hotel_name": "NORTHSTAR HOTEL BERLIN",
    "hotel_address": "Alexanderplatz 99, 10178 Berlin, Germany",
    "invoice_number": "NSB-2026-0914-1042",
    "invoice_date": "2026-09-14",
    "guest_name": "Alex Morgan",
    "check_in_date": "2026-09-10",
    "check_out_date": "2026-09-14",
    "number_of_nights": 4,
    "currency": "EUR",
    "room_rate": 140.00,
    "city_tax": 42.00,
    "total_amount": 712.60,
    "payment_method": "Visa ending 4242"
}

### 4. Define the Extraction Prompt

In [162]:
schema_json = json.dumps(
    HotelInvoiceExtraction.model_json_schema(),
    indent=2
)

prompt = f"""
You are extracting structured information from a hotel invoice.

Return only valid JSON that follows exactly this schema:

{schema_json}

Rules:
- Return every field defined in the schema.
- Use only information explicitly present in the document.
- Do not infer missing values.
- Use null if a value is not available.
- Preserve monetary values as numbers.
- Use ISO date format YYYY-MM-DD where possible.
- Return JSON only.

Document:
{text}
"""

### 5. Configure Models

In [163]:
MODELS = [
    "gemma3:12b",
    "qwen3:14b",
]

### 6. Define Model Execution and validate Model Output

In [164]:
def call_model(model_name: str, prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        },
        timeout=120,
    )

    response.raise_for_status()

    return response.json()["response"]

In [165]:
def parse_model_output(output: str) -> dict:
    cleaned = output.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]

    elif cleaned.startswith("```"):
        cleaned = cleaned[len("```"):]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    return json.loads(cleaned)


def validate_extracted_data(extracted_data: dict) -> HotelInvoiceExtraction:
    return HotelInvoiceExtraction.model_validate(extracted_data)

### 7. Run Models

In [166]:
results = {}

for model_name in MODELS:
    print(f"Running {model_name}...")

    try:
        start_time = time.perf_counter()

        raw_output = call_model(
            model_name=model_name,
            prompt=prompt,
        )

        duration_seconds = time.perf_counter() - start_time

        extracted_data = parse_model_output(raw_output)

        validated_result = validate_extracted_data(
            extracted_data
        )

        validated_data = validated_result.model_dump()

        results[model_name] = {
            "raw_output": raw_output,
            "extracted_data": validated_data,
            "duration_seconds": duration_seconds,
        }

        print(
            f"Done in {duration_seconds:.2f} seconds."
        )

    except Exception as e:
        print(f"{model_name} failed: {e}")

Running gemma3:12b...
Done in 17.12 seconds.
Running qwen3:14b...
Done in 38.77 seconds.


### 8. Display Extracted Results

In [167]:
for model_name, result in results.items():
    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    print(f"Duration: {result['duration_seconds']:.2f} seconds\n")

    print(
        json.dumps(
            result["extracted_data"],
            indent=2,
            ensure_ascii=False
        )
    )


MODEL: gemma3:12b
Duration: 17.12 seconds

{
  "hotel_name": "NORTHSTAR HOTEL BERLIN",
  "hotel_address": "Alexanderplatz 99, 10178 Berlin, Germany",
  "invoice_number": "NSB-2026-0914-1042",
  "invoice_date": "2026-09-14",
  "guest_name": "Alex Morgan",
  "check_in_date": "2026-09-10",
  "check_out_date": "2026-09-14",
  "number_of_nights": 4,
  "currency": "EUR",
  "room_rate": 140.0,
  "city_tax": 42.0,
  "total_amount": 712.6,
  "payment_method": "Visa ending 4242"
}

MODEL: qwen3:14b
Duration: 38.77 seconds

{
  "hotel_name": "NORTHSTAR HOTEL BERLIN",
  "hotel_address": "Alexanderplatz 99, 10178 Berlin, Germany",
  "invoice_number": "NSB-2026-0914-1042",
  "invoice_date": "2026-09-14",
  "guest_name": "Alex Morgan",
  "check_in_date": "2026-09-10",
  "check_out_date": "2026-09-14",
  "number_of_nights": 4,
  "currency": "EUR",
  "room_rate": 140.0,
  "city_tax": 42.0,
  "total_amount": 712.6,
  "payment_method": "Visa ending 4242"
}


### 9. Compare with Ground Truth

In [168]:
evaluation_results = {}

for model_name, result in results.items():
    extracted_data = result["extracted_data"]

    comparison_rows = []

    for field, expected_value in ground_truth.items():
        actual_value = extracted_data.get(field)

        comparison_rows.append({
            "field": field,
            "expected": expected_value,
            "actual": actual_value,
            "match": expected_value == actual_value,
        })

    comparison_df = pd.DataFrame(comparison_rows)

    matches = int(comparison_df["match"].sum())
    total = len(comparison_df)
    accuracy = matches / total

    evaluation_results[model_name] = {
        "comparison": comparison_df,
        "matches": matches,
        "total": total,
        "accuracy": accuracy,
    }

### 10. Display Comparison Tables

In [169]:
for model_name, evaluation in evaluation_results.items():
    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    print(
        f"Correct fields: "
        f"{evaluation['matches']}/{evaluation['total']} "
        f"({evaluation['accuracy']:.2%})"
    )

    display(evaluation["comparison"])


MODEL: gemma3:12b
Correct fields: 13/13 (100.00%)


,field,expected,actual,match
0,hotel_name,NORTHSTAR HOTEL BERLIN,NORTHSTAR HOTEL BERLIN,True
1,hotel_address,"Alexanderplatz 99, 10178 Berlin, Germany","Alexanderplatz 99, 10178 Berlin, Germany",True
2,invoice_number,NSB-2026-0914-1042,NSB-2026-0914-1042,True
3,invoice_date,2026-09-14,2026-09-14,True
4,guest_name,Alex Morgan,Alex Morgan,True
5,check_in_date,2026-09-10,2026-09-10,True
6,check_out_date,2026-09-14,2026-09-14,True
7,number_of_nights,4,4,True
8,currency,EUR,EUR,True
9,room_rate,140.0,140.0,True



MODEL: qwen3:14b
Correct fields: 13/13 (100.00%)


,field,expected,actual,match
0,hotel_name,NORTHSTAR HOTEL BERLIN,NORTHSTAR HOTEL BERLIN,True
1,hotel_address,"Alexanderplatz 99, 10178 Berlin, Germany","Alexanderplatz 99, 10178 Berlin, Germany",True
2,invoice_number,NSB-2026-0914-1042,NSB-2026-0914-1042,True
3,invoice_date,2026-09-14,2026-09-14,True
4,guest_name,Alex Morgan,Alex Morgan,True
5,check_in_date,2026-09-10,2026-09-10,True
6,check_out_date,2026-09-14,2026-09-14,True
7,number_of_nights,4,4,True
8,currency,EUR,EUR,True
9,room_rate,140.0,140.0,True


### 11. Compare Model Performance

In [170]:
summary_df = pd.DataFrame([
    {
        "model": model_name,
        "correct_fields": evaluation["matches"],
        "total_fields": evaluation["total"],
        "accuracy": evaluation["accuracy"],
        "duration_seconds": results[model_name]["duration_seconds"],
    }
    for model_name, evaluation in evaluation_results.items()
])

summary_display = summary_df.copy()

summary_display["accuracy"] = summary_display["accuracy"].map(
    lambda value: f"{value:.2%}"
)

summary_display["duration_seconds"] = summary_display["duration_seconds"].map(
    lambda value: f"{value:.2f}"
)

display(summary_display)

,model,correct_fields,total_fields,accuracy,duration_seconds
0,gemma3:12b,13,13,100.00%,17.12
1,qwen3:14b,13,13,100.00%,38.77


### 12. Analyze Extraction Errors

In [171]:
for model_name, evaluation in evaluation_results.items():
    errors_df = evaluation["comparison"][
        ~evaluation["comparison"]["match"]
    ]

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    if errors_df.empty:
        print("No extraction errors.")
    else:
        display(errors_df)


MODEL: gemma3:12b
No extraction errors.

MODEL: qwen3:14b
No extraction errors.


### 13. Observations

Both tested models successfully extracted the target fields from the
machine-readable hotel invoice.

The extracted outputs were validated against the expected data structure
and compared with manually defined ground-truth values.

The experiment records both field-level extraction accuracy and model
processing time.

Further experiments should evaluate:
- different hotel invoice layouts,
- missing or ambiguous fields,
- additional travel document types,
- scanned documents requiring OCR,
- robustness across a larger test dataset.